# Phase 2: exploring the images

**CRISP-DM stage:** Data understanding

**Runs on:** Kaggle, with the GPU and internet switched off, because no neural network runs and nothing is installed. The private dataset `revathyh/mvtec-ad2-sheet-metal-vial` is attached.

**Figures with dataset images:** files named `gallery_*` hold real dataset images, so they stay on my laptop and out of git. Each figure is saved and closed, not displayed, so no image ends up in the notebook's log.

## 1. Find the data

Same search as the Phase 1 notebook, see its part 1 for why we search and do not type a fixed path.

In [ ]:
from pathlib import Path

INPUT_FOLDER = Path("/kaggle/input")
PRODUCT_NAMES = ["sheet_metal", "vial"]

product_folders = {}

for train_folder in INPUT_FOLDER.rglob("train"):
    product_folder = train_folder.parent
    if product_folder.name in PRODUCT_NAMES:
        product_folders[product_folder.name] = product_folder

for product_name in PRODUCT_NAMES:
    if product_name not in product_folders:
        raise FileNotFoundError("Could not find the folder for " + product_name)
    print(product_name, "folder:", product_folders[product_name])

## 2. Helper functions

**Matplotlib** is Python's standard plotting library. A figure holds a grid of panels called axes, and each image goes in one panel.

- `pick_evenly` chooses files spread through a sorted folder. This shows more variety than the first few files, and it repeats exactly with no randomness.
- `draw_image` draws one greyscale image. `vmin=0` and `vmax=255` fix the brightness scale, so images can be compared. Matplotlib's default stretches each image's own range, which would make a dark image look normal.
- `contour` draws a line where the mask's values cross a level. 127.5 is halfway between 0 (no defect) and 255 (defect), so the line follows the defect's edge. An outline keeps the defect visible, where a filled overlay would hide it.

In [ ]:
import matplotlib.pyplot as plt
import numpy
from PIL import Image

# Sharp enough for the report, at a reasonable file size
FIGURE_DPI = 150


def pick_evenly(paths, how_many):
    """Return up to how_many paths, spread evenly through the list."""
    if len(paths) <= how_many:
        return paths
    picked = []
    step = len(paths) / how_many
    for position in range(how_many):
        picked.append(paths[int(position * step)])
    return picked


def read_image(path):
    """Read a PNG file into a table of numbers."""
    with Image.open(path) as image:
        return numpy.array(image)


def draw_image(axis, image_path, title, mask_path=None):
    """Draw one greyscale image in one panel, with its mask outlined if given."""
    axis.imshow(read_image(image_path), cmap="gray", vmin=0, vmax=255)
    if mask_path is not None:
        axis.contour(read_image(mask_path), levels=[127.5], colors="red", linewidths=1)
    axis.set_title(title, fontsize=10)
    axis.axis("off")


def mask_path_for(product_folder, image_path):
    """Return the mask file that belongs to a defective test image."""
    mask_name = image_path.stem + "_mask.png"
    return product_folder / "test_public" / "ground_truth" / "bad" / mask_name

## 3. Sample gallery

Three rows per product: good training images, good validation images, and defective test images with the mask outlined in red. There are 4 images per row, enough to see variety while staying legible. The defective row uses `regular` lighting, so it shows the defect and not a lighting change.

The figure size differs per product, because Sheet Metal is four times wider than tall and Vial is taller than wide.

**Limit:** the figure is shrunk to fit, so the tiniest defects may be hard to see, even with an outline.

In [ ]:
IMAGES_PER_ROW = 4
GALLERY_SIZES = {"sheet_metal": (24, 6), "vial": (16, 17)}

for product_name in PRODUCT_NAMES:
    product_folder = product_folders[product_name]

    train_paths = sorted((product_folder / "train" / "good").glob("*.png"))
    validation_paths = sorted((product_folder / "validation" / "good").glob("*.png"))
    defective_paths = sorted((product_folder / "test_public" / "bad").glob("*_regular.png"))

    rows = [
        ("train", pick_evenly(train_paths, IMAGES_PER_ROW)),
        ("validation", pick_evenly(validation_paths, IMAGES_PER_ROW)),
        ("defective test", pick_evenly(defective_paths, IMAGES_PER_ROW)),
    ]

    # squeeze=False keeps the panels in a table of rows and columns
    figure, axes = plt.subplots(3, IMAGES_PER_ROW, figsize=GALLERY_SIZES[product_name], squeeze=False)

    for row_number in range(3):
        row_name, paths = rows[row_number]
        for column_number in range(IMAGES_PER_ROW):
            axis = axes[row_number][column_number]
            # A row can have fewer images than panels on a small test sample
            if column_number >= len(paths):
                axis.axis("off")
                continue
            image_path = paths[column_number]
            mask_path = None
            if row_name == "defective test":
                mask_path = mask_path_for(product_folder, image_path)
            draw_image(axis, image_path, row_name + ": " + image_path.name, mask_path)

    figure.suptitle("MVTec AD 2, " + product_name + " (CC BY-NC-SA 4.0). Red outline: defect mask")
    file_name = "gallery_" + product_name + ".png"
    figure.savefig(file_name, dpi=FIGURE_DPI, bbox_inches="tight")
    # Closing without showing keeps dataset images out of the notebook's log
    plt.close(figure)
    print("Saved", file_name)

## 4. Scene check

File names suggest each number in `test_public` is one scene photographed under several lighting conditions. Here we look at defective scene `000` under every condition, with its masks outlined. This is one scene per product, so it can support the idea but cannot prove it for every scene.

In [ ]:
SCENE_NUMBER = "000"
SCENE_COLUMNS = {"sheet_metal": 2, "vial": 4}
SCENE_SIZES = {"sheet_metal": (24, 10), "vial": (16, 11)}

for product_name in PRODUCT_NAMES:
    product_folder = product_folders[product_name]
    scene_paths = sorted((product_folder / "test_public" / "bad").glob(SCENE_NUMBER + "_*.png"))

    columns = SCENE_COLUMNS[product_name]
    # Enough rows to hold every image, rounding up
    number_of_rows = (len(scene_paths) + columns - 1) // columns
    figure, axes = plt.subplots(number_of_rows, columns, figsize=SCENE_SIZES[product_name], squeeze=False)

    for row_number in range(number_of_rows):
        for column_number in range(columns):
            axis = axes[row_number][column_number]
            position = row_number * columns + column_number
            if position >= len(scene_paths):
                axis.axis("off")
                continue
            image_path = scene_paths[position]
            draw_image(axis, image_path, image_path.name, mask_path_for(product_folder, image_path))

    figure.suptitle("MVTec AD 2, " + product_name + ", scene " + SCENE_NUMBER + " (CC BY-NC-SA 4.0). Red outline: defect mask")
    file_name = "gallery_scene_check_" + product_name + ".png"
    figure.savefig(file_name, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(figure)
    print("Saved", file_name, "with", len(scene_paths), "images")